## Inheritance (Paretnt -> Child classes)
Inheritance allows a new class (called child or subclass ) to automatically reuse all attributes and methods of an exisiting class (called a parent or superclass), while adding its own specialized behaviour


- syntax = class ChildClass(ParentClass)

- super(): Calls a method (most commonly __init__) from the parent class , so we don't over rewrite duplicate setup code

- Method overriding = When a child class deifines a method with the exact same name as the parent, the child's version replaces the paren'ts behaviour for objects of that child class

### Example 1 - E commerce Payment Gateways

Every payment methods shares an amount of currency plus a receipt formatter, while each specific gateway processesthe charge differently

In [5]:
class PaymentMethod:
    def __init__(self, amount: float, currency: str = "INR") -> None:
        self.amount = amount
        self.currency  = currency
    
    def generate_receipt(self) -> str:
        return f"Receipt: Paid {self.currency} {self.amount:.2f}"
    
    def process_payment(self) -> str:
        raise NotImplementedError("Subclasses must implement process_payment()")


class UPIPayment(PaymentMethod):

    def __init__(self, amount:float, upi_id:str, currency: str = "INR") -> None:
        super().__init__(amount, currency) # Reuses Parent's __init__
        self.upi_id = upi_id
    
    # Overriding parent method with UPI_specific logic

    def process_payment(self) -> None:
        return f"Debiting {self.currency} {self.amount:.2f} via UPI { self.upi_id}"


class CreditCardPayment(PaymentMethod):
    
    def __init__(self, amount: float, card_last4: str, currency: str = "INR") -> None:
        super().__init__(amount,currency)
        self.card_last4 = card_last4
    
    # Overriding parent method with a card- specific
    def process_payment(self) -> str:
        return f"charging {self.currency} {self.amount:.2f} to card ending in {self.card_last4}"



upi = UPIPayment(1499.0,"devanath@0ksbi")
card = CreditCardPayment(4500.0, "8821")

print(upi.process_payment()) # Child's overridden method
print(upi.generate_receipt()) # Ineherited directly from PaymentMethod
print(card.process_payment())


Debiting INR 1499.00 via UPI devanath@0ksbi
Receipt: Paid INR 1499.00
charging INR 4500.00 to card ending in 8821


## Extending Parent Behaviour with super()

sometimes we don't want to completely replace a parent's method - we want to run the parent's logic first and then ad extra steps on top.

### Example 2: Clud storage quota and Tiered Accounts

In [7]:
class CloudAccount:

    def __init__(self, username: str, storage_limit_gb: int = 15) -> None:
        self.username = username
        self.storage_limit_gb = storage_limit_gb
        self.used_gb = 0.0

    def upload_file(self, size_gb:float) -> str:
        if self.used_gb + size_gb > self.storage_limit_gb:
            return f"Upload failed: Exceeds  { self.storage_limit_gb}GB limit"
        self.used_gb += size_gb
        return f"Uploaded {size_gb}GB. Used: {self.used_gb}/{self.storage_limit_gb}GB"



class EnterpriseCloudAccount(CloudAccount):
    def __init__(self, username: str, company:str) -> None:
        super().__init__(username, storage_limit_gb=1000)
        self.company = company
        self.audit_logs : list[str] = []

    
    # Runs parent's upload_file() via super(), then records an audit trail

    def upload_file(self, size_gb:float) -> str:
        result = super().upload_file(size_gb)
        self.audit_logs.append(f"[{self.company}] {self.username}: {result}")
        return result


corp_user = EnterpriseCloudAccount("devanath", "AchmeCorp")
print(corp_user.upload_file(120.5))
print("Audit trail:", corp_user.audit_logs)

Uploaded 120.5GB. Used: 120.5/1000GB
Audit trail: ['[AchmeCorp] devanath: Uploaded 120.5GB. Used: 120.5/1000GB']


## Polymorphism ("Many forms")

Polymorphism means different classes share the same method interface so our calling code can treat all of them uniformly without writing

if isinstance(x, UPIPayment): ... elif isinstance(x,CreditCardPayment): ..


### Example 3: Multi-Channel Notification Dispatcher

Notice how send_bulk_alrets() loops through a mixed list of EmailAlert, SMSAlert, and SlackAlert objects and simply calls .send() on each-each object know how to execute its own version

In [8]:
class EmailAlert:
    def __init__(self, recipient: str) -> None:
        self.recipient = recipient
    
    def send(self, message: str) -> str:
        return f"[EMAIL -> {self.recipient}] Subject: Alert | Body: {message}"


class SMSAlert:
    def __init__(self, phone: str) -> None:
        self.phone = phone
    
    def send(self, message:str) -> None:
        return f"[SMS -> {self.phone}] {message[:40]}"
    

class SlackAlert:
    def __init__(self, channel: str) -> None:
        self.channel = channel
    
    def send(self, message: str) -> str:
        return f"[SLACK {self.channel}] : rotating_light: {message}"



# Polymorphic function: works with ANY object that implements .send(message)

def dispatch_incident(channels: list[EmailAlert | SMSAlert | SlackAlert], alert_text:str) -> None:
    for channel in channels:
        print(channel.send(alert_text))
    

subscribers = [
    EmailAlert("onceall@company.com"),
    SMSAlert("+91-987654321"),
    SlackAlert("#prod-incidents"),
]

dispatch_incident(subscribers, "Database CPU usage exceeded 92% for 5 minutes.")

[EMAIL -> onceall@company.com] Subject: Alert | Body: Database CPU usage exceeded 92% for 5 minutes.
[SMS -> +91-987654321] Database CPU usage exceeded 92% for 5 mi
[SLACK #prod-incidents] : rotating_light: Database CPU usage exceeded 92% for 5 minutes.


**Note** : Python also supports Duck Typing - EmailAlert, SMSAlert, and SlackAlert didn't even need to inherit from a common parent class for dispatch_incident() to work, as long as they all provide .send()

## Useful Built-in Type Checks( isinstance & issubclass)



In [10]:
print(isinstance(upi, UPIPayment))
print(isinstance(upi, PaymentMethod)) # True - because a child is alos an instance of its parent
print(issubclass(UPIPayment, PaymentMethod))

True
True
True


### Your Turn


In [2]:
class Employee:
    def __init__(self, name: str, base_salary: float) -> None:
        self.name = name
        self.base_salary = base_salary

    def calculate_pay(self) -> float:
        return self.base_salary


class FullTimeEmployee(Employee):
    def __init__ (self, name: str, bonus: float, base_salary: float) -> None:
        super().__init__(name,base_salary) # reusing the parent class
        self.bonus = bonus

    def calculate_pay(self) -> float:
        base_sal = super().calculate_pay()
        return base_sal + self.bonus
    

class ContractEmployee(Employee):
    def __init__(self,name: str, hourly_rate: float, hours_worked: int) -> None:
        super().__init__(name,base_salary=0.0)
        self.hourly_rate = hourly_rate
        self.hours_worked = hours_worked
    
    def calculate_pay(self) -> float:
        return self.hourly_rate * self.hours_worked


employees = [FullTimeEmployee("devan",100.0,2000.0),
ContractEmployee("Suku",12.0,5)]

for employee in employees:
    print(employee.calculate_pay())

2100.0
60.0
